In [95]:
import random, math, time

Clasa care defineste un cromozom

In [96]:
class Cromozom:
    NR_GENE=20
    PROBABILITATE_MUTATIE=0.15
    PROCENT_SELECTIE=0.5
    NR_MAX_PASI=5000
    NR_CROMOZOMI = 150

    PROCENT_CROMO_PASTRATI=10
    PROCENT_CROMO_SELECTIE=60
    
    def __init__(self, info=None):
        self.info=info if info else [random.randint(0,1) for _ in range(Cromozom.NR_GENE)]
        self.scor=self.fitness()


    def fitness(self):
        """Funcția care evaluează scorul cromozomului
        Va fi suprascrisă mai jos

        Returns:
            float: scorul cromozomului
        """
        return 0

    @classmethod
    def selectie(cls, cromozomi: list) -> list:
        """
        Selectia pentru evolutie

        Args:
            cromozomi (list): Lista de cromozomi din care alegem "părinții"
        """
        
        cromozomi_selectati=list(cromozomi)
        random.shuffle(cromozomi_selectati)
        cromozomi_selectati=cromozomi_selectati[:Cromozom.NR_CROMOZOMI//2]
        cromozomi_selectati.sort(key=lambda  x: x.scor, reverse=True)
        cromozomi_selectati=cromozomi_selectati[:1+math.floor(Cromozom.PROCENT_CROMO_SELECTIE/100*len(cromozomi_selectati))]
        return cromozomi_selectati


    def mutatie(self,info_cromozom):
        for i in  range(Cromozom.NR_GENE):
            if random.random() < Cromozom.PROBABILITATE_MUTATIE:
                info_cromozom[i]=1-info_cromozom[i]
                
                
    def evolutie(self,cromozom2):
        #recombinare
        info_cromozom_nou=[self.info[i] if random.random() < 0.5 
            else cromozom2.info[i] 
            for i in range(Cromozom.NR_GENE)]
        #mutatie
        self.mutatie(info_cromozom_nou)
        return Cromozom(info_cromozom_nou)
    
    def __str__(self):
        return " ".join(map(str,self.info))+f"\nScor:{self.scor}"

    def __repr__(self):
        return f"Informatie:{self.info}\nScor:{self.scor}"


In [97]:
class CromozomMatriceal(Cromozom):
    N_LIN=8
    N_COL=10
    
    def __init__(self, info=None):
        self.info=info if info else [[random.randint(0,1) for _ in range(CromozomMatriceal.N_COL)] for _ in range(CromozomMatriceal.N_LIN)]
        self.scor=self.fitness()

    def mutatie(self,info_cromozom):
        for i in range(CromozomMatriceal.N_LIN):
            for j in range(CromozomMatriceal.N_COL):
                if random.random()< Cromozom.PROBABILITATE_MUTATIE:
                    info_cromozom[i][j]=1-info_cromozom[i][j]
    
    def evolutie(self,cromozom2):
        info_cromozom_nou=[]
        for i in range(CromozomMatriceal.N_LIN):
            linie=[self.info[i][j] 
                if random.random()<0.5 
                else cromozom2.info[i][j]
                for j in range(CromozomMatriceal.N_COL)]
            info_cromozom_nou+=[linie]

        self.mutatie(info_cromozom_nou)

        return CromozomMatriceal(info_cromozom_nou)
    

    
    def __str__(self):
        sir=""
        for i in range(CromozomMatriceal.N_LIN):
            sir+=" ".join(map(str,self.info[i]))+"\n"
        sir+=f"\nScor:{self.scor}"
        return sir

In [98]:

def algoritmGenetic(ClsCromozom:type[Cromozom]):
    NR_PASI_AFIS=500
    t1=time.time()
    #crearea populatiei cu cromozomi aleatori
    cromozomi=[ClsCromozom() for _ in range(ClsCromozom.NR_CROMOZOMI)]
    for pas in range(ClsCromozom.NR_MAX_PASI):
        cromozomi_noi=[]
        #sortam cromozomii dupa fitness
        cromozomi.sort(key=lambda  x: x.scor, reverse=True)
        #alegem o parte din cei mai buni cromozomi si-i adaugam in populatia noua
        cromozomi_noi+=cromozomi[:1+math.floor((ClsCromozom.PROCENT_CROMO_PASTRATI/100)*ClsCromozom.NR_CROMOZOMI)]
        cromozomi_selectati=ClsCromozom.selectie(cromozomi)
        for i in range(ClsCromozom.NR_CROMOZOMI-len(cromozomi_noi)):
            x1,x2=random.choices(population=cromozomi_selectati,k=2)
            cromozomi_noi+=[x1.evolutie(x2)]
        cromozomi=cromozomi_noi
        if pas%NR_PASI_AFIS==0:
            print(f"Pas {pas}. Fitness total: {fitness_total(cromozomi)}")
    rezultat=max(cromozomi, key=lambda  x: x.scor)
    t2=time.time()
    print(f"Solutie:\n{rezultat}")
    print(f"Timp: {round(t2-t1,4)} secunde")
    

def fitness_total(cromozomi):
    return sum([x.scor for x in cromozomi])


Funcția de fitness în cazul în care dorim să obținem un vector cu valori alternate.

In [99]:
def fitness1(self):
    suma=0
    for i in range(len(self.info)-1):
        suma+=(self.info[i]!=self.info[i+1])
    return suma

Cromozom.fitness=fitness1
algoritmGenetic(Cromozom)

Pas 0. Fitness total: 1481
Pas 500. Fitness total: 1855
Pas 1000. Fitness total: 1824
Pas 1500. Fitness total: 1882
Pas 2000. Fitness total: 1811
Pas 2500. Fitness total: 1874
Pas 3000. Fitness total: 1801
Pas 3500. Fitness total: 1861
Pas 4000. Fitness total: 1883
Pas 4500. Fitness total: 1864
Solutie:
0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1 0 1
Scor:19
Timp: 2.766 secunde


Funcția de fitness în cazul în care dorim să obținem ca soluții vectori conținând cât mai multe valori de 1

In [100]:
def fitness2(self):
    return sum(self.info)
Cromozom.fitness=fitness2
algoritmGenetic(Cromozom)


Pas 0. Fitness total: 1730
Pas 500. Fitness total: 2343
Pas 1000. Fitness total: 2262
Pas 1500. Fitness total: 2302
Pas 2000. Fitness total: 2325
Pas 2500. Fitness total: 2301
Pas 3000. Fitness total: 2334
Pas 3500. Fitness total: 2276
Pas 4000. Fitness total: 2331
Pas 4500. Fitness total: 2260
Solutie:
1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
Scor:20
Timp: 2.2884 secunde


Funcția de fitness în cazul în care dorim să obținem ca soluții vectori conținând cât mai multe valori de 0

In [101]:
def fitness3(self):
    return Cromozom.NR_GENE-sum(self.info)
Cromozom.fitness=fitness3
algoritmGenetic(Cromozom)

Pas 0. Fitness total: 1668
Pas 500. Fitness total: 2277
Pas 1000. Fitness total: 2286
Pas 1500. Fitness total: 2347
Pas 2000. Fitness total: 2297
Pas 2500. Fitness total: 2299
Pas 3000. Fitness total: 2380
Pas 3500. Fitness total: 2299
Pas 4000. Fitness total: 2358
Pas 4500. Fitness total: 2327
Solutie:
0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
Scor:20
Timp: 2.3593 secunde


Soluția sa fie un vector în care avem doar K valori de 1, cu K dat (eventual ca parametru), cele K valori de 1 pot fi oriunde în vector

In [102]:

K=5
# TO DO


Cromozomul solutie, matriceal, trebuie sa aiba celulele cu valori alternate de 0 și 1 (vecinii unui 0 ar fi preferabil să fie 1, iar vecinii lui 1 ar fi preferabil să fie 0). Consideram vecinii doar pe linie si coloana.

De exemplu:<br/>
1010101010<br/>
0101010101<br/>
1010101010<br/>
0101010101<br/>
1010101010<br/>
0101010101<br/>
1010101010<br/>
<br/>
sau <br/>
0101010101<br/>
1010101010<br/>
0101010101<br/>
1010101010<br/>
0101010101<br/>
1010101010<br/>
0101010101


In [103]:
def fitness_mat1(self):
    suma=0
    for i in range(CromozomMatriceal.N_LIN):
        for j in  range(CromozomMatriceal.N_COL):
            if i<CromozomMatriceal.N_LIN-1:
                suma+=(self.info[i][j]!=self.info[i+1][j])
            if j<CromozomMatriceal.N_COL-1:
                suma+=(self.info[i][j]!=self.info[i][j+1])
    return suma

CromozomMatriceal.fitness=fitness_mat1
algoritmGenetic(CromozomMatriceal)

Pas 0. Fitness total: 11011
Pas 500. Fitness total: 11873
Pas 1000. Fitness total: 12095
Pas 1500. Fitness total: 12066
Pas 2000. Fitness total: 12151
Pas 2500. Fitness total: 12357
Pas 3000. Fitness total: 12477
Pas 3500. Fitness total: 12559
Pas 4000. Fitness total: 12254
Pas 4500. Fitness total: 12325
Solutie:
1 0 1 0 1 0 0 1 0 1
0 1 0 1 0 1 0 0 1 0
1 0 1 0 1 0 1 1 0 1
0 1 0 1 0 1 1 0 1 0
1 0 1 0 1 0 1 1 0 1
0 1 0 1 0 1 1 0 1 0
1 0 1 1 1 0 0 1 0 1
0 1 0 0 0 1 0 0 1 1

Scor:122
Timp: 15.8353 secunde


Cromozomul solutie ar trebui sa aiba 1 doar pe prima/ultima linie, respectiv prima si ultima coloana si 0 in restul matricii.
De exemplu:<br/>
1111111111<br/>
1000000001<br/>
1000000001<br/>
1000000001<br/>
1000000001<br/>
1000000001<br/>
1111111111<br/>


In [104]:

def fitness_mat2(self):
    suma=0
    for i in  range(CromozomMatriceal.N_LIN):
        for j in  range(CromozomMatriceal.N_COL):
            if i==0 or i==CromozomMatriceal.N_LIN-1 or j==0 or j==CromozomMatriceal.N_COL-1:
                if self.info[i][j]==1:
                    suma+=1
            else:
                if self.info[i][j]==0:
                    suma+=1
    return suma

CromozomMatriceal.fitness=fitness_mat2
algoritmGenetic(CromozomMatriceal)


Pas 0. Fitness total: 6308
Pas 500. Fitness total: 8192
Pas 1000. Fitness total: 8477
Pas 1500. Fitness total: 8228
Pas 2000. Fitness total: 8165
Pas 2500. Fitness total: 8182
Pas 3000. Fitness total: 8279
Pas 3500. Fitness total: 8306
Pas 4000. Fitness total: 8341
Pas 4500. Fitness total: 8364
Solutie:
1 1 1 1 1 1 1 1 1 1
1 0 0 0 0 0 0 0 0 1
1 0 0 0 0 0 0 0 0 1
1 0 0 1 0 0 0 0 0 1
1 0 0 0 0 0 1 0 0 1
1 0 1 0 0 0 0 0 0 1
1 0 0 1 0 1 0 0 0 1
1 1 1 1 1 1 0 1 1 1

Scor:74
Timp: 12.8612 secunde


Cromozomul solutie ar trebui sa aiba 1 inspre margini si 0 central.
Exemplu de cromozom acceptabil:<br/>
1111111111<br/>
1111001111<br/>
1110000011<br/>
1100000011<br/>
1100000111<br/>
1111000111<br/>
1111111111


In [105]:

def fitness_mat3(self):
    suma=0
    centru_i=CromozomMatriceal.N_LIN//2
    centru_j=CromozomMatriceal.N_COL//2
    RAZA=3 #depinde de problema
    for i in  range(CromozomMatriceal.N_LIN):
        for j in  range(CromozomMatriceal.N_COL):
            dist=math.sqrt((centru_i-i)*(centru_i-i)+(centru_j-j)*(centru_j-j))
            if dist<=RAZA:
                if self.info[i][j]==0:
                    suma+=1
            else:
                if self.info[i][j]==1:
                    suma+=1
    return suma


CromozomMatriceal.fitness=fitness_mat3
algoritmGenetic(CromozomMatriceal)

Pas 0. Fitness total: 6454
Pas 500. Fitness total: 8220
Pas 1000. Fitness total: 8104
Pas 1500. Fitness total: 8368
Pas 2000. Fitness total: 8181
Pas 2500. Fitness total: 8339
Pas 3000. Fitness total: 8021
Pas 3500. Fitness total: 8328
Pas 4000. Fitness total: 8390
Pas 4500. Fitness total: 8638
Solutie:
1 1 1 1 1 1 1 1 1 0
1 1 1 1 1 0 1 1 1 1
1 1 1 0 0 0 0 0 1 1
1 1 1 0 0 0 0 0 0 0
1 1 0 0 0 0 0 1 0 1
1 1 1 0 0 0 0 0 1 1
1 1 1 0 0 0 0 1 1 1
1 1 1 1 1 0 1 1 1 1

Scor:75
Timp: 15.839 secunde


Considerăm cromozomul tăiat la jumătate pe orizontală respectiv verticală în 4 submatrici egale. Cromozomul solutie ar trebui sa aibă submatricele din stânga-sus și dreapta-jos conținând valori de 0, iar submatrciele din dreapta sus și stânga-jos continând valori de 1. 
Exemplu de cromozom acceptabil:<br/>
0000011111<br/>
0000011111<br/>
0000011111<br/>
0000011111<br/>
1111100000<br/>
1111100000<br/>
1111100000<br/>
1111100000


In [106]:
# TO DO 2